# FloodWatch KP: Historical Case Evaluation Harness (2010 & 2022 Floods)
This notebook runs rigorous external case evaluations for the **2010 Pakistan Super Floods** and the **2022 Pakistan Monsoon Floods**.

### Key Rules:
1. **Separate Ground Truth and Predictions:** Official source-confirmed impact areas are reported separately from model predictions. Model predictions are NEVER presented as confirmed historical events.
2. **Comparable Areas Only:** Performance metrics (POD, FAR, CSI, F1, Brier score) are computed strictly over basins where spatial granularity and physical mechanisms support comparison.
3. **Transparent Non-Comparable / Unmapped Reporting:** Basins located outside the convective flood zone or lacking point streamgauge data are explicitly cataloged with scientific caveats.

In [ ]:
# Setup environment & automatic Google Colab repository cloning
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    if not (Path.cwd() / 'src' / 'floodwatch').exists():
        if not Path('FLASH_FLOOD_PREDICTION').exists():
            !git clone -b qamran/real-data-eval https://github.com/Syed-Muhammad-Qamran/FLASH_FLOOD_PREDICTION.git
        %cd FLASH_FLOOD_PREDICTION
    !pip install -q pyyaml pyarrow

repo_root = Path.cwd()
if not (repo_root / 'src' / 'floodwatch').exists() and (repo_root.parent / 'src' / 'floodwatch').exists():
    repo_root = repo_root.parent

if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

print(f'Repository Root: {repo_root}')


In [ ]:
# Inspect Official Citations & Provenance
citations = historical_events.list_citations()
print("Official Reference Citations:")
for cit in citations:
    print(f"- {cit['name']} ({cit['date']})")
    print(f"  Granularity: {cit['granularity']} | Attribution: {cit['attribution']}")
    print(f"  URL: {cit['url']}\n")

In [ ]:
# Run External Evaluation on 2010 & 2022 Cases
for year in [2010, 2022]:
    # Check if real model predictions or demo predictions exist
    demo_p = repo_root / "artifacts" / "a" / "predictions_a.parquet"
    if not demo_p.exists():
        demo_p = repo_root / "runs" / "demo" / "predictions_fused.parquet"
        
    df_preds = pd.read_parquet(demo_p)
    if "prob" not in df_preds.columns:
        df_preds["prob"] = df_preds["prob_a"]
        
    report = eval_cases.evaluate_case(case_year=year, predictions_df=df_preds, model_name="Model A", threshold=0.50)
    
    print("=" * 85)
    print(f"HISTORICAL CASE {year}: {report.title.upper()}")
    print(f"Window: {report.window_start} to {report.window_end} | Mechanism: {report.mechanism.upper()}")
    print("=" * 85)
    
    m = report.metrics
    print(f"Verification Metrics (Comparable Basins):")
    print(f"  POD (Recall):        {m['pod']:.3f} (Detection Rate)")
    print(f"  FAR (False Alarm):   {m['far']:.3f}")
    print(f"  CSI (Critical Succ): {m['csi']:.3f}")
    print(f"  F1-Score:            {m['f1']:.3f}")
    print(f"  Brier Score:         {m['brier_score']:.3f}")
    print(f"  Contingency:         TP={int(m['tp'])}, FP={int(m['fp'])}, FN={int(m['fn'])}, TN={int(m['tn'])}\n")
    
    comp_df = pd.DataFrame([
        {
            "Basin": c.basin_id,
            "Official Status": c.confirmed_status.title(),
            "Spatial Granularity": c.spatial_granularity.title(),
            "Model Alert": "ALERT" if c.model_predicted_risk else "NORMAL",
            "Max Prob": f"{c.model_max_prob:.3f}",
            "Tier": c.model_tier,
            "Outcome": c.comparison_outcome,
            "Caveat / Mapping": c.caveat,
        }
        for c in report.comparisons
    ])
    display(comp_df)
    print("\n")